<a href="https://colab.research.google.com/github/madhumitha006/Madhumitha-Codeboosters-internship-2026/blob/main/Day_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI agents vs chatbots

Tool calling+Multi step reasoning

text to sql

What is ai agent?

AI agent is a techinal agent who checks your calender,

searches online sends an email and calls the vendor
all from one request

multi step planning

what is tool calling?

Tool calling means giving the ai the ability to run specific python

react = reasoning + acting the agent loops through four steps:

*Think

*plan

*act

*Respond

technical architecture

1.schema injection

2.sql genaration

3.execute+respond

The complex text-to-sql pipeline

load csv into sql lite

get database schema

generate sql withgroq

execute sql on sql lite

In [ ]:
!pip install groq -q
print("Libraries installed successfully")

Libraries installed successfully


In [ ]:
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("All libraries imported successfully")

All libraries imported successfully


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"
client = Groq(api_key = os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"
print("Groq client initialized succesfully")
print(f"Using model:{MODEL}")

Groq client initialized succesfully
Using model:llama-3.1-8b-instant


In [ ]:
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
df = pd.read_csv(io.StringIO(csv_data))
print(f"Dataset loaded:{len(df)} rows,{len(df.columns)} columns")
print("\nFirst 5 rows:")
df.head()

Dataset loaded:30 rows,8 columns

First 5 rows:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
conn = sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("Database created:college.db")
print("Table 'students' created with 30 students records")
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f"\nVerification:{test_df['total_rows'][0]} rows in database")

Database created:college.db
Table 'students' created with 30 students records

Verification:30 rows in database


In [ ]:
def get_schema(conn,table_name="students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"Table:{table_name}"]
  schema_lines.append("Columns:")
  for col in columns:
    schema_lines.append(f" - {col[1]}({col[2]})")
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()
  schema_lines.append("\nSample rows(first 3):")
  for row in sample_rows:
    schema_lines.append(f" - {row}")
  return "\n".join(schema_lines)
schema = get_schema(conn)
print(schema)

Table:students
Columns:
 - student_id(INTEGER)
 - name(TEXT)
 - age(INTEGER)
 - gender(TEXT)
 - subject(TEXT)
 - marks(INTEGER)
 - attendance(INTEGER)
 - grade(TEXT)

Sample rows(first 3):
 - (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 - (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 - (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question, schema_text, client, model):
    system_prompt = f"""
You are an expert SQL assistant.

You are connected to a SQLite database with the following structure:
{schema_text}

Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text - only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is: students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Math')
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question}
        ],
        temperature=0.0
    )

    sql_query = response.choices[0].message.content.strip()
    return sql_query


question = "Show me all female students"

print(f"Question: {question}")
print("\nGenerating SQL....")

sql = generate_sql(question, schema, client, MODEL)

print(f"\nGenerated SQL:\n{sql}")

Question: Show me all female students

Generating SQL....

Generated SQL:
SELECT * FROM students WHERE gender = 'Female'


In [ ]:
import re
import pandas as pd

def execute_sql(sql_query, conn):
    clean_sql = sql_query.strip()
    clean_sql = re.sub(r"```sql\s*", "", clean_sql)
    clean_sql = re.sub(r"```", "", clean_sql)
    clean_sql = clean_sql.strip()
    try:
        result_df = pd.read_sql_query(clean_sql, conn)
        return result_df, None
    except Exception as e:
        return None, str(e)
print(f"Executing SQL:{sql}")
result,error=execute_sql(sql,conn)
if error:
    print(f"Error:{error}")
else:
    print(f"\nQuery returned{len(result)} rows:")
    print(result)

Executing SQL:SELECT * FROM students WHERE gender = 'Female'

Query returned15 rows:
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female  M

In [ ]:
def text_to_sql_agent(user_question, conn, client, model, verbose=True):
    print("=" * 60)
    print(f"User Question: {user_question}")
    print("=" * 60)

    if verbose:
        print("\n[STEP 1] Reading database schema...")

    schema_text = get_schema(conn)

    if verbose:
        print("Schema loaded successfully")

    if verbose:
        print("\n[STEP 2] Generating SQL query with Groq LLM...")

    generated_sql = generate_sql(user_question, schema_text, client, model)

    if verbose:
        print(f"Generated SQL:\n{generated_sql}")

    if verbose:
        print("\n[STEP 3] Executing SQL on the database...")

    result_df, error = execute_sql(generated_sql, conn)

    if error:
        print(f"SQL Execution Error: {error}")
        return None, generated_sql

    if verbose:
        print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
        print("\nRESULTS:")
        print("-" * 40)
        print(result_df.to_string(index=False))
        print("=" * 60)

    return result_df, generated_sql


result, sql_used = text_to_sql_agent(
    "Show top 5 students in Programming",
    conn,
    client,
    MODEL
)

User Question: Show top 5 students in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT name, marks, attendance, grade FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
        name  marks  attendance grade
Aditya Kumar     97          99    A+
 Rohan Mehta     95          98    A+
Anjali Singh     94          97    A+
 Nandita Rao     92          96    A+
  Arjun Nair     91          94    A+


In [ ]:
result1, _ = text_to_sql_agent(
    "Show all students who study Mathematics",
    conn,client,MODEL
)

User Question: Show all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Femal

In [ ]:
result2, _ = text_to_sql_agent(
    "What is the average marks for each subject?",
    conn,client,MODEL
)

User Question: What is the average marks for each subject?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT subject, AVG(marks) FROM students GROUP BY subject

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 3 row(s)

RESULTS:
----------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4


In [ ]:
result3, _ = text_to_sql_agent(
    "Show students who scored more than 90 marks",
    conn,client,MODEL
)

User Question: Show students who scored more than 90 marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE marks > 90

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 6 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+


In [ ]:
result4, _ = text_to_sql_agent(
    "How many male and female students are there?",
    conn,client,MODEL
)

User Question: How many male and female students are there?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT COUNT(*) FROM students WHERE gender = 'Male' AND gender = 'Female'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 COUNT(*)
        0


In [ ]:
result5, _ = text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming,ordered by marks",
    conn,client,MODEL
)

User Question: Show female students who scored above 85 in Science or Programming,ordered by marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE gender = 'Female' AND (subject = 'Science' OR subject = 'Programming') AND marks > 85 ORDER BY marks

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         14 Kavitha Rajan   21 Female Programming     86          93     A
          8  Ananya Gupta   21 Female Programming     89          96     A
         26   Nandita Rao   21 Female Programming     92          96    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         20  Anjali Singh   21 Female Programming     94          97    A+


In [ ]:
def generate_answer(user_question, result_df, client, model):

    result_text = result_df.to_string(index=False)

    prompt = f"""
User Question:
{user_question}

SQL Result:
{result_text}

Provide a concise natural language answer based on the SQL result.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful data analyst."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()